# Sugarcane Anomaly Early Warning System
Cell-by-cell Sentinel-1 RVI workflow. Run from the project root. The synthetic section proves the pipeline before real data are substituted.

## 1. Install libraries
For Conda, use `environment.yml`. In a clean Jupyter kernel run the next cell once.

In [ ]:
%pip install -q numpy pandas geopandas rasterio shapely pyproj pyogrio scipy scikit-learn statsmodels matplotlib plotly streamlit pyyaml xarray "dask[distributed]"

## 2. Imports and project paths

In [ ]:
from pathlib import Path
import sys, numpy as np, pandas as pd, geopandas as gpd, rasterio
import matplotlib.pyplot as plt
from shapely.geometry import box
sys.path.insert(0,str(Path.cwd()))
from src.rvi_calculation import calculate_rvi, rvi_geotiff
from src.baseline_model import fit_healthy_baseline, attach_baseline
from src.anomaly_detection import score_rules, isolation_forest
from src.pixel_anomaly import anomaly_raster
from src.visualization import save_risk_png
OUT=Path('outputs'); (OUT/'rasters').mkdir(parents=True,exist_ok=True)

## 3. Preprocessing rule
Raw Sentinel-1 GRD SAFE scenes require an external SNAP graph. For each scene: apply orbit, border/thermal noise removal, radiometric calibration, optional multi-temporal or Refined Lee speckle filtering, terrain flattening/correction, then export co-registered VV/VH. Keep units and orbit direction explicit. Do not apply the RVI formula to dB values.

In [ ]:
from src.preprocessing import run_snap_gpt
# Example only, after SNAP and an exported graph are installed:
# run_snap_gpt('scene.SAFE','data/processed/scene.tif','config/s1_preprocess.xml',gpt='/opt/snap/bin/gpt')

## 4. Synthetic calibrated VV/VH GeoTIFF smoke test
Replace these paths with real paired images after the demonstration.

In [ ]:
height=width=180
transform=rasterio.transform.from_origin(34.80,-15.80,0.0001,0.0001)
crs='EPSG:4326'; rng=np.random.default_rng(42)
y,x=np.mgrid[:height,:width]
vv_db=-9+rng.normal(0,.7,(height,width)); vh_db=-15+rng.normal(0,.8,(height,width))
# Inject a low-VH stress patch
stress=(x-105)**2+(y-80)**2<28**2; vh_db[stress]-=5
profile=dict(driver='GTiff',height=height,width=width,count=1,dtype='float32',crs=crs,transform=transform,nodata=-9999,compress='deflate')
for name,a in [('VV',vv_db),('VH',vh_db)]:
    with rasterio.open(OUT/'rasters'/f'demo_{name}.tif','w',**profile) as ds: ds.write(a.astype('float32'),1)
rvi_geotiff(OUT/'rasters/demo_VV.tif',OUT/'rasters/demo_VH.tif',OUT/'rasters/demo_RVI.tif',units='db')

## 5. Display RVI

In [ ]:
with rasterio.open(OUT/'rasters/demo_RVI.tif') as ds: rvi=ds.read(1,masked=True)
plt.figure(figsize=(9,7)); plt.imshow(rvi,cmap='viridis',vmin=0,vmax=1.5); plt.colorbar(label='RVI'); plt.title('Sentinel-1 RVI'); plt.axis('off'); plt.show()

## 6. Healthy baseline and block anomaly features
In production, `observed_rvi` comes from zonal statistics over every date. Healthy labels come from audited seasons.

In [ ]:
dates=pd.date_range('2023-01-01',periods=18,freq='14D'); rows=[]
for year in [2023,2024,2025]:
  for block in ['A01','A02','A12']:
    for age in range(0,252,14):
      expected=.25+.65*np.sin(np.pi*min(age,250)/300)
      obs=expected+rng.normal(0,.035)
      healthy=year<2025 or block!='A12'
      if year==2025 and block=='A12' and age>84: obs-=.24
      rows.append(dict(block_id=block,date=pd.Timestamp(year,1,1)+pd.Timedelta(days=age),planting_date=pd.Timestamp(year,1,1),variety='NCo376',crop_type='ratoon',healthy=healthy,observed_rvi=obs,is_problem=int(not healthy)))
obs=pd.DataFrame(rows)
baseline=fit_healthy_baseline(obs,min_samples=4); joined=attach_baseline(obs,baseline); scored=score_rules(joined); scored,if_model=isolation_forest(scored)
scored.tail()

## 7. Healthy versus observed curve

In [ ]:
latest=scored[scored.date.dt.year==2025]; b='A12'; q=latest[latest.block_id==b]
plt.figure(figsize=(10,5)); plt.plot(q.crop_age_days,q.observed_rvi,'o-',label='Observed'); plt.plot(q.crop_age_days,q.expected_rvi,label='Healthy expected'); plt.fill_between(q.crop_age_days,q.expected_rvi-2*q.expected_std,q.expected_rvi+2*q.expected_std,alpha=.2,label='95% envelope'); plt.xlabel('Crop age (days)'); plt.ylabel('RVI'); plt.legend(); plt.grid(alpha=.2); plt.show()

## 8. Create and display the categorical anomaly heat map
The expected value below would normally be selected per pixel/block crop age. The output GeoTIFF has an embedded QGIS colour table.

In [ ]:
risk_path=anomaly_raster(OUT/'rasters/demo_RVI.tif',expected=.78,expected_std=.08,out_path=OUT/'rasters/latest_risk.tif')
fig=save_risk_png(risk_path,OUT/'figures_latest_risk.png',title='Sugarcane RVI anomaly heat map'); plt.show()

## 9. Download underlying GeoTIFF from Jupyter

In [ ]:
from IPython.display import FileLink, display
display(FileLink(str(risk_path)))
print('QGIS-ready file:',risk_path.resolve())

## 10. Real-data loop

In [ ]:
# Naming example: data/raw/vv/2025-01-12_VV.tif and data/raw/vh/2025-01-12_VH.tif
# for vv_path in sorted(Path('data/raw/vv').glob('*_VV.tif')):
#     date=vv_path.name[:10]; vh_path=Path('data/raw/vh')/vv_path.name.replace('_VV','_VH')
#     rvi_geotiff(vv_path,vh_path,Path('data/processed')/f'{date}_RVI.tif',units='db')

## 11. Run dashboard
From a terminal in this project: `streamlit run dashboard.py`. The app exposes report, GeoJSON, and GeoTIFF downloads.